# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [18]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [19]:
import os
import chromadb
from dotenv import load_dotenv

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool


In [20]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [21]:
@tool
def retrieve_game(query: str):
    """
    Semantic search: Finds most relevant games in the vector DB
    args:
    - query: a question about game industry.
    
    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    chroma_client = chromadb.PersistentClient(path="chromadb")
    collection = chroma_client.get_collection("udaplay")
    
    results = collection.query(
        query_texts=[query],
        n_results=5
    )
    
    print(f"ChromaDB query results: {results}") 
    metadatas = results.get('metadatas', [[]])[0] if results else []
    
    return [
        {
            "Platform": m.get("Platform"),
            "Name": m.get("Name"),
            "YearOfRelease": m.get("YearOfRelease"),
            "Description": m.get("Description")
        }
        for m in metadatas
    ]

#### Evaluate Retrieval Tool

In [22]:
import ast
import json as _json
from pydantic import BaseModel, Field
from lib.parsers import PydanticOutputParser


class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Detailed explanation of the evaluation result")


def _parse_docs(retrieved_docs_str: str) -> list:
    """
    Parse the retrieved_docs argument, which the LLM sends as a repr/JSON string
    of the list returned by retrieve_game.
    """
    if isinstance(retrieved_docs_str, list):
        return retrieved_docs_str  # already a proper list
    for parser in (_json.loads, ast.literal_eval):
        try:
            result = parser(retrieved_docs_str)
            if isinstance(result, list):
                return result
            if isinstance(result, dict):
                return [result]
        except Exception:
            continue
    return []


@tool
def evaluate_retrieval(question: str, retrieved_docs: str):
    """
    Based on the user's question and on the list of retrieved documents, 
    it will analyze the usability of the documents to respond to that question.
    args: 
    - question: original question from user
    - retrieved_docs: JSON or repr string of the list of retrieved documents from retrieve_game
    
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    docs = _parse_docs(retrieved_docs)

    print(f"\n[DEBUG] parsed {len(docs)} doc(s) from retrieved_docs")
    for i, d in enumerate(docs[:3]):
        print(f"[DEBUG] doc[{i}]: {d}")

    docs_text = "\n".join([
        f"- Name: {d.get('Name', 'Unknown')} | Platform: {d.get('Platform', 'Unknown')} | Release Year: {d.get('YearOfRelease', 'Unknown')} | Description: {d.get('Description', '')}"
        for d in docs
    ])

    prompt = f"""Your task is to evaluate if the documents are enough to respond the query.
Give a detailed explanation, so it's possible to take an action to accept it or not.

Query: {question}

Retrieved Documents:
{docs_text}"""

    llm = LLM(api_key=OPENAI_API_KEY)
    response = llm.invoke(prompt, response_format=EvaluationReport)

    parser = PydanticOutputParser(model_class=EvaluationReport)
    report = parser.parse(response)

    return {"useful": report.useful, "description": report.description}

#### Game Web Search Tool

In [23]:
from tavily import TavilyClient


@tool
def game_web_search(question: str):
    """
    Web search: Finds relevant results on the web about the game industry.
    Use this when the vector DB does not have enough information to answer the question.
    args:
    - question: a question about game industry.
    """
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(query=question, max_results=5)

    return [
        {
            "title": result.get("title"),
            "url": result.get("url"),
            "content": result.get("content")
        }
        for result in response.get("results", [])
    ]

### Agent

In [24]:
SYSTEM_INSTRUCTIONS = """You are UdaPlay, an expert AI research agent for the video game industry.

Your goal is to answer questions accurately about video games using the following workflow:
1. Use `retrieve_game` to search the internal knowledge base first.
2. Use `evaluate_retrieval` to assess whether the retrieved documents are sufficient.
   - Pass the original question and the full result from `retrieve_game` as a string.
   - `evaluate_retrieval` returns an EvaluationReport with two fields:
       * useful (bool): true if the documents contain enough information to answer the question.
       * description (str): explanation of why the documents are or are not sufficient.
   - If useful is true, proceed directly to synthesizing an answer from the retrieved documents.
   - If useful is false, you MUST call `game_web_search` before answering.
3. Use `game_web_search` only when evaluate_retrieval returns useful=false.
4. Synthesize all available information into a clear, accurate, and concise answer.

Guidelines:
- Always try the internal knowledge base before searching the web.
- Trust the EvaluationReport: if useful=true, do NOT call game_web_search.
- Be factual and cite sources when using web results.
- If no information is found anywhere, say so clearly.
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=SYSTEM_INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.3,
)

In [25]:
import json as _json_trace

def run_with_trace(question: str):
    """Run a single query and surface every tool call + evaluation verdict."""
    print(f"Q: {question}")
    print("-" * 65)

    run = agent.invoke(question)
    state = run.get_final_state()

    tools_called = []
    web_search_triggered = False

    for msg in state["messages"]:
        # Print every tool the LLM decided to call
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for call in msg.tool_calls:
                tools_called.append(call.function.name)
                print(f"  → Tool called : {call.function.name}")

        # Surface the evaluation verdict so the decision process is visible
        if isinstance(msg, ToolMessage) and msg.name == "evaluate_retrieval":
            try:
                import ast as _ast
                payload = _json_trace.loads(msg.content)  # unwrap outer JSON string
                # payload may be a Python repr string – try json then ast
                if isinstance(payload, str):
                    try:
                        ev = _json_trace.loads(payload)
                    except Exception:
                        ev = _ast.literal_eval(payload)
                else:
                    ev = payload
                useful = ev.get("useful", "?")
                desc = str(ev.get("description", ""))[:120]
                print(f"  → Evaluation  : useful={useful}")
                print(f"                  {desc}...")
            except Exception:
                print(f"  → Evaluation  : (raw) {msg.content[:120]}")

        if isinstance(msg, ToolMessage) and msg.name == "game_web_search":
            web_search_triggered = True

    answer = next(
        (m.content for m in reversed(state["messages"])
         if isinstance(m, AIMessage) and m.content),
        "No answer found."
    )
    print(f"A: {answer}")
    print(f"   [web fallback triggered: {web_search_triggered}]")
    print()
    return {"question": question, "tools": tools_called, "web_fallback": web_search_triggered}


# Query 1: Pokémon Gold/Silver  – present in local DB (Game Boy Color era games)
# Query 2: Mortal Kombat X on PS5 – dataset has only 15 games; not present → web fallback expected
# Query 3: First 3D Mario platformer – Super Mario 64 should be in local DB
queries = [
    "When was Pokémon Gold and Silver released?",
    "Was Mortal Kombat X released for PlayStation 5?",
    "Which was the first 3D platformer Mario game?",
]

trace_results = [run_with_trace(q) for q in queries]

Q: When was Pokémon Gold and Silver released?
-----------------------------------------------------------------
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
ChromaDB query results: {'ids': [['006', '007', '012', '009', '008']], 'embeddings': None, 'documents': [['[Game Boy Color] Pokémon Gold and Silver (1999) - Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.', '[Game Boy Advance] Pokémon Ruby and Sapphire (2002) - Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.', '[Nintendo Switch] Mario Kart 8 Deluxe (2017) - An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.', "[Nintendo 64] Super Mario 64 (1996) - A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.", '[Super Nintendo Entertainment System (SNES)] 

### Performance Summary

| # | Query | Tools Used | Web Fallback |
|---|-------|------------|--------------|
| 1 | When was Pokémon Gold and Silver released? | retrieve_game → evaluate_retrieval | No |
| 2 | Was Mortal Kombat X released for PlayStation 5? | retrieve_game → evaluate_retrieval → game_web_search | **Yes** |
| 3 | Which was the first 3D platformer Mario game? | retrieve_game → evaluate_retrieval | No |

**Observations:**
- **Query 1 & 3** were answered entirely from the local vector DB. The evaluation step confirmed the retrieved documents were sufficient (`useful=true`), so no web search was triggered.
- **Query 2** demonstrates the fallback path: the 15-game dataset does not contain Mortal Kombat X, so `evaluate_retrieval` returned `useful=false` and the agent automatically invoked `game_web_search` to supplement with current web results.
- The agent's workflow (retrieve → evaluate → optional web search → synthesise) worked as designed: it minimises unnecessary web calls while gracefully handling gaps in the local knowledge base.

### (Optional) Advanced

In [26]:
from lib.memory import LongTermMemory, MemoryFragment
from lib.vector_db import VectorStoreManager
from lib.state_machine import StateMachine, Step, EntryPoint, Termination
from lib.agents import AgentState

import json


# ── Long-term memory setup ──────────────────────────────────────────────────

vector_store_manager = VectorStoreManager(openai_api_key=OPENAI_API_KEY)
long_term_memory = LongTermMemory(db=vector_store_manager)

USER_ID = "udaplay_user"


@tool
def save_to_memory(fact: str):
    """
    Save an important fact or insight to long-term memory for future reference.
    args:
    - fact: a fact or insight worth remembering about the game industry.
    """
    fragment = MemoryFragment(content=fact, owner=USER_ID, namespace="game_facts")
    long_term_memory.register(fragment)
    return {"saved": True, "content": fact}


@tool
def recall_from_memory(query: str):
    """
    Search long-term memory for previously stored facts relevant to the query.
    args:
    - query: a question or topic to search for in long-term memory.
    """
    result = long_term_memory.search(query_text=query, owner=USER_ID, namespace="game_facts", limit=3)
    return [fragment.content for fragment in result.fragments]


def make_tool_node(tool_fn):
    """Wrap a tool into a state machine Step that runs all pending calls for it."""
    def step_logic(state: AgentState) -> AgentState:
        tool_calls = state.get("current_tool_calls") or []
        tool_messages = []

        for call in tool_calls:
            if call.function.name != tool_fn.name:
                continue
            args = json.loads(call.function.arguments)
            result = str(tool_fn(**args))
            tool_messages.append(
                ToolMessage(content=json.dumps(result), tool_call_id=call.id, name=tool_fn.name)
            )

        return {
            "messages": state["messages"] + tool_messages,
            "current_tool_calls": None,
            "session_id": state["session_id"],
        }

    return Step[AgentState](tool_fn.name, step_logic)


extended_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=SYSTEM_INSTRUCTIONS + "\nYou also have access to `save_to_memory` and `recall_from_memory` tools. "
                 "Use `recall_from_memory` at the start of each query, and `save_to_memory` when you learn a new fact.",
    tools=[retrieve_game, evaluate_retrieval, game_web_search, save_to_memory, recall_from_memory],
    temperature=0.3,
)

# Verify the state machine nodes are wired correctly
print("Agent state machine steps:", list(extended_agent.workflow.steps.keys()))
print("Long-term memory ready for user:", USER_ID)

Agent state machine steps: ['__entry__', 'message_prep', 'llm_processor', 'tool_executor', '__termination__']
Long-term memory ready for user: udaplay_user


### Multi-Turn Conversation (Stateful)

The agent maintains conversation history per `session_id`. When the same `session_id` is reused, the previous messages are restored from `ShortTermMemory` and passed to the next invocation — enabling context-aware follow-up questions.

In [27]:
def get_answer(run) -> str:
    """Extract the last non-empty AI response from a run."""
    state = run.get_final_state()
    return next(
        (m.content for m in reversed(state["messages"]) if isinstance(m, AIMessage) and m.content),
        "No answer found."
    )

# All turns share the same session_id so history is preserved across calls
session_id = "test_user_1234"

# Turn 1 – ask about a specific game
turn1_query = "Tell me about The Witcher 3"
run1 = agent.invoke(turn1_query, session_id=session_id)
answer1 = get_answer(run1)
print(f"Turn 1 | Q: {turn1_query}")
print(f"       | A: {answer1}")

# Turn 2 – follow-up with no explicit game name; relies on prior context
turn2_query = "What genre is it?"
run2 = agent.invoke(turn2_query, session_id=session_id)
answer2 = get_answer(run2)
print(f"Turn 2 | Q: {turn2_query}")
print(f"       | A: {answer2}\n")

# Show how many messages are now carried in the session
final_state = run2.get_final_state()
print(f"Messages in session '{session_id}': {len(final_state['messages'])}")
print("(Includes system prompt, both user turns, all assistant replies, and any tool messages)")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
ChromaDB query results: {'ids': [['003', '004', '009', '012', '014']], 'embeddings': None, 'documents': [['[PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.', "[PlayStation 4] Marvel's Spider-Man (2018) - An open-world superhero game that lets players swing through New York City as Spider-Man, battling iconic villains.", "[Nintendo 64] Super Mario 64 (1996) - A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.", '[Nintendo Switch] Mario Kart 8 Deluxe (2017) - An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.', '[Xbox One] Minecraft (2014) - A sandbox game that allows players to build and explore infinite worlds, fostering creativity and adve